# Lezione 12 — Caso d'uso integrativo: dai dati grezzi a un modello predittivo

**Obiettivi della lezione:**
- Eseguire una pipeline completa di pulizia, analisi esplorativa e modellazione di
  riferimento su un dataset reale.
- Collegare ogni passo della pipeline alle sei categorie di attività della Lezione 1,
  chiudendo il cerchio del corso.

Basato sul capitolo 21 del manuale e adattato da `noaa-weather-data-v1.ipynb`
(correggendo 3 problemi di compatibilità del notebook originale: `np.bool`
eliminato da NumPy moderno, `fillna(method=...)` obsoleto in pandas moderno, e un
`!!pip install` con doppio segno).

## Il caso d'uso

Dati meteorologici orari reali dell'aeroporto JFK di New York (114.546
osservazioni, licenza CDLA-Sharing). Obiettivo: aiutare a programmare meglio gli
orari dei voli anticipando la temperatura. Percorriamo le tre fasi del
capitolo 2: **pulizia**, **esplorazione (EDA)** e **modellazione**.

In [ ]:
!pip install --quiet pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 15)


## Parte 1 — Pulizia dei dati

In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fJKLoMKtgpDAl5MkA6rf2A/jfk-weather.csv"
crudo = pd.read_csv(url, parse_dates=["DATE"], low_memory=False)

# Teniamo un sottoinsieme di colonne rilevanti per il caso d'uso
columnas = ["DATE", "HOURLYDRYBULBTEMPF", "HOURLYPrecip",
            "HOURLYWindSpeed", "HOURLYRelativeHumidity", "HOURLYStationPressure"]
clima = crudo[columnas].copy()
clima = clima.set_index(pd.DatetimeIndex(clima["DATE"])).drop(columns=["DATE"])

print(f"Filas antes de limpiar: {len(clima)}")
clima.head()


In [ ]:
# La colonna di precipitazione usa 'T' per "traccia" (quasi nulla) e '*' per dato mancante
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].replace("T", "0.00")
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].astype(str).str.replace("s", "", regex=False)

# Il resto delle colonne: '*' significa dato mancante
for columna in clima.columns:
    clima[columna] = pd.to_numeric(
        clima[columna].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )

# Riempimento dei buchi in avanti (correzione del "fillna(method=...)" obsoleto:
# si usa direttamente .ffill())
clima = clima.ffill()
clima = clima.dropna()

print(f"Filas después de limpiar: {len(clima)}")
clima.describe()


## Parte 2 — Analisi esplorativa (EDA)

In [ ]:
# Tendenza della temperatura durante il 2017
clima.loc["2017", "HOURLYDRYBULBTEMPF"].plot(figsize=(10, 4))
plt.title("Temperatura horaria en JFK durante 2017")
plt.ylabel("°F")
plt.show()


In [ ]:
# Mappa di correlazione tra variabili
# (correzione del bug originale: si usa dtype=bool invece di np.bool, eliminato
# da NumPy moderno)
correlacion = clima.corr()
mascara = np.zeros_like(correlacion, dtype=bool)
mascara[np.triu_indices_from(mascara)] = True

plt.figure(figsize=(6, 5))
sns.heatmap(correlacion, mask=mascara, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlación entre variables meteorológicas")
plt.show()


## Parte 3 — Un primo modello predittivo (baseline)

Prima di qualsiasi modello sofisticato, si confronta con un modello di riferimento molto
semplice: "la prossima ora avrà la stessa temperatura dell'ora attuale".

In [ ]:
from sklearn.metrics import mean_squared_error

temperatura = clima["HOURLYDRYBULBTEMPF"]

# Baseline: prevedere "nessun cambiamento" rispetto all'ora precedente
prediccion_baseline = temperatura.shift(1).dropna()
real = temperatura.loc[prediccion_baseline.index]

rmse_baseline = np.sqrt(mean_squared_error(real, prediccion_baseline))
print(f"RMSE del modelo de referencia (sin cambio): {rmse_baseline:.2f} °F")


In [ ]:
# Un modello appena più elaborato: media delle ultime 3 ore
prediccion_media_movil = temperatura.rolling(3).mean().shift(1).dropna()
real_2 = temperatura.loc[prediccion_media_movil.index]

rmse_media_movil = np.sqrt(mean_squared_error(real_2, prediccion_media_movil))
print(f"RMSE del modelo de media móvil (3 horas): {rmse_media_movil:.2f} °F")

if rmse_media_movil < rmse_baseline:
    print("La media móvil mejora al modelo de referencia.")
else:
    print("La media móvil NO mejora al modelo de referencia.")


## Riepilogo e chiusura del corso

- Hai percorso il ciclo completo: gestione dei dati -> pulizia (ETL) -> visualizzazione
  (EDA) -> modellazione -> valutazione — esattamente le sei attività della Lezione 1.
- Un progetto reale quasi mai inizia dal modello più sofisticato: inizia dal
  pulire i dati e confrontare con un modello di riferimento molto semplice.
- Con questo si conclude il percorso del corso. Tutto ciò che hai usato qui (Python, pandas,
  API, notebook, Git/GitHub) è la stessa cassetta degli attrezzi comune a
  praticamente qualsiasi progetto reale di dati.

Continua con il notebook di pratica di questa lezione per chiudere il corso con un
esercizio tuo su questo stesso dataset.